In [92]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
import time
import pickle

In [93]:
train_df = pd.read_csv("../data/train.csv")
feature_df = pd.read_csv("../data/features.csv")
stores_df = pd.read_csv("../data/stores.csv")

In [94]:
df = pd.merge(train_df,feature_df,on=['Store','Date','IsHoliday'],how='left')

df = pd.merge(df,stores_df,on = ['Store'],how = 'left')

df['Date'].min(),df['Date'].max()
df.columns.tolist()

['Store',
 'Dept',
 'Date',
 'Weekly_Sales',
 'IsHoliday',
 'Temperature',
 'Fuel_Price',
 'MarkDown1',
 'MarkDown2',
 'MarkDown3',
 'MarkDown4',
 'MarkDown5',
 'CPI',
 'Unemployment',
 'Type',
 'Size']

In [95]:
df['MarkDown1'] = df['MarkDown1'].fillna(0)
df['MarkDown2'] = df['MarkDown2'].fillna(0)
df['MarkDown3'] = df['MarkDown3'].fillna(0)
df['MarkDown4'] = df['MarkDown4'].fillna(0)
df['MarkDown5'] = df['MarkDown5'].fillna(0)

print("Missing Values in MarkDown Columns:")
print(df[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].isnull().sum())

Missing Values in MarkDown Columns:
MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64


In [96]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['WeekOfYear'] = df['Date'].dt.isocalendar().week

df['IsHoliday'] = df['IsHoliday'].astype(int)

df = df.drop(columns = ['Date'])

df.head()

,Store,Dept,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Year,Month,WeekOfYear
0,1,1,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,2010,2,5
1,1,1,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,2010,2,6
2,1,1,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,2010,2,7
3,1,1,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,2010,2,8
4,1,1,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,2010,3,9


In [97]:
x = df.drop(columns = ['Weekly_Sales'])
y = df['Weekly_Sales']

train_mask = (df['Year'] < 2012) | ((df['Year'] == 2012) & (df['Month'] < 7))
test_mask = ~train_mask

train_df = df[train_mask]
test_df = df[test_mask]

x_train = train_df.drop(columns = ['Weekly_Sales'])
x_test = test_df.drop(columns = ['Weekly_Sales'])

y_train = train_df['Weekly_Sales']
y_test = test_df['Weekly_Sales']


print(f"Safe time-split complete!")
print(f"X train shape: {x_train.shape}")
print(f"X_test Shape: {x_test.shape}")

Safe time-split complete!
X train shape: (371242, 17)
X_test Shape: (50328, 17)


In [98]:
numeric_features = ['Store','Dept','IsHoliday','Temperature','Fuel_Price','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5','CPI','Unemployment','Size','Year','Month','WeekOfYear']
categorical_features = ['Type']

numeric_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot',OneHotEncoder(sparse_output=False,handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num',numeric_transformer,numeric_features),
    ('cat',categorical_transformer,categorical_features)
])

print("Memory Blueprint (Preprocessor) is ready!")

Memory Blueprint (Preprocessor) is ready!


In [102]:
model_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',RandomForestRegressor(n_estimators=50,random_state=42,n_jobs=-1))
])

print("Master Pipeline built! Starting Training....")
start_time = time.time()
model_pipeline.fit(x_train,y_train)
print(f"Training Completed in {round(time.time() - start_time) / 60,2} Minutes!")

print("Predicting the test set")

pred = model_pipeline.predict(x_test)

mae = mean_absolute_error(y_test,pred)
r2_score = r2_score(y_test,pred)

print("Mean Absolute Error:",round(mae,2))
print("R2 Score:",round(r2_score,2))

Master Pipeline built! Starting Training....
Training Completed in (0.6333333333333333, 2) Minutes!
Predicting the test set


TypeError: 'float' object is not callable

In [100]:
with open("../model/sales_model.pkl","wb") as f:
    pickle.dump(model_pipeline,f)

print("Pipeline succesfully Saved to disk!")

Pipeline succesfully Saved to disk!
